## build network dataset

In [1]:
import numpy as np
import pandas as pd
import node2vec
import matplotlib.pyplot as plt

C:\Users\davel\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fdemand = pd.read_csv('../../data/data/processed/trips.csv')

In [3]:
station = pd.read_csv('../../data/data/processed/stations.csv')

In [4]:
station_clean = station.drop(['street_address'],axis = 1)

In [5]:
weather = pd.read_csv('../../data/data/processed/weather.csv')

In [6]:
fdemand.head()

,duration,start_station,end_station,plan_duration,st_year,st_month,st_day,st_hour,st_minute,end_year,...,pass_walkup,trip_oneway,day_of_week,is_weekend,st_hr_sin,st_hr_cos,st_day_sin,st_day_cos,st_yr_sin,st_yr_cos
0,11,3049,3072,30,2020,1,1,0,13,2020,...,0,1,2,0,0.0,1.0,0.0,1.0,0.0,1.0
1,28,3124,3053,30,2020,1,1,0,14,2020,...,0,1,2,0,0.0,1.0,0.0,1.0,0.0,1.0
2,28,3124,3053,30,2020,1,1,0,14,2020,...,0,1,2,0,0.0,1.0,0.0,1.0,0.0,1.0
3,21,3005,3018,30,2020,1,1,0,17,2020,...,0,1,2,0,0.0,1.0,0.0,1.0,0.0,1.0
4,21,3005,3018,30,2020,1,1,0,17,2020,...,0,1,2,0,0.0,1.0,0.0,1.0,0.0,1.0


### 1. Creating the Station Network Graph

I'll create a directed graph where each unique station is a node, and each trip from a `start_station` to an `end_station` forms a directed edge. Since we're interested in station properties, I'll aggregate the trips to count the number of connections between stations to form edge weights. If a trip exists from A to B, an edge `A -> B` is added.

In [7]:
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add nodes for all unique stations
all_stations = pd.concat([fdemand['start_station'], fdemand['end_station']]).unique()
G.add_nodes_from(all_stations)

# Aggregate trips to get edge weights (number of trips between stations)
edges = fdemand.groupby(['start_station', 'end_station']).size().reset_index(name='weight')

# Add edges with weights to the graph
for _, row in edges.iterrows():
    G.add_edge(row['start_station'], row['end_station'], weight=row['weight'])

print(f"Number of nodes (stations): {G.number_of_nodes()}")
print(f"Number of edges (trips between stations): {G.number_of_edges()}")

Number of nodes (stations): 345
Number of edges (trips between stations): 77508


### 2. Calculating Centrality Measures

Now, I'll compute various centrality measures for each station in the network. These measures help identify the most important or influential stations based on different network properties.

#### a. Degree Centrality

**Definition**: Degree centrality measures the number of direct connections a node has. In a directed graph, we distinguish between **in-degree** (number of incoming connections) and **out-degree** (number of outgoing connections).

*   **In-degree centrality**: The number of times a station is an `end_station`.
*   **Out-degree centrality**: The number of times a station is a `start_station`.

In [8]:
# Calculate In-Degree Centrality
in_degree_centrality = nx.in_degree_centrality(G)
in_degree_df = pd.DataFrame(in_degree_centrality.items(), columns=['station', 'in_deg'])

# Calculate Out-Degree Centrality
out_degree_centrality = nx.out_degree_centrality(G)
out_degree_df = pd.DataFrame(out_degree_centrality.items(), columns=['station', 'out_deg'])

# Combine into a single DataFrame for all centralities
station_centralities = pd.merge(in_degree_df, out_degree_df, on='station')

print("Top 5 stations by In-Degree Centrality:")
display(station_centralities.sort_values(by='in_deg', ascending=False).head())
print("\nTop 5 stations by Out-Degree Centrality:")
display(station_centralities.sort_values(by='out_deg', ascending=False).head())

Top 5 stations by In-Degree Centrality:


,station,in_deg,out_deg
136,3000,0.979651,0.093023
41,3010,0.938953,0.915698
74,3185,0.924419,0.930233
104,3057,0.921512,0.930233
166,3255,0.909884,0.906977



Top 5 stations by Out-Degree Centrality:


,station,in_deg,out_deg
80,3059,0.906977,0.930233
74,3185,0.924419,0.930233
104,3057,0.921512,0.930233
39,3054,0.904070,0.924419
101,3022,0.898256,0.921512


#### b. Betweenness Centrality

**Definition**: Betweenness centrality quantifies the number of times a node acts as a bridge along the shortest path between two other nodes. Nodes with high betweenness centrality are important for information flow within the network, as they lie on many communication paths.

In [9]:
# Calculate Betweenness Centrality
betweenness_centrality = nx.betweenness_centrality(G)
betweenness_df = pd.DataFrame(betweenness_centrality.items(), columns=['station', 'betweenness'])

station_centralities = pd.merge(station_centralities, betweenness_df, on='station')

print("Top 5 stations by Betweenness Centrality:")
display(station_centralities.sort_values(by='betweenness', ascending=False).head())

Top 5 stations by Betweenness Centrality:


,station,in_deg,out_deg,betweenness
68,3074,0.880814,0.886628,0.004104
26,3102,0.886628,0.901163,0.003564
101,3022,0.898256,0.921512,0.003540
92,3112,0.837209,0.851744,0.003429
20,3160,0.840116,0.883721,0.003349


#### c. Closeness Centrality

**Definition**: Closeness centrality measures how 'close' a node is to all other nodes in the network. It's calculated as the reciprocal of the sum of the shortest path distances from a node to all other reachable nodes. A node with high closeness centrality can quickly reach all other nodes.

In [10]:
# Calculate Closeness Centrality (note: for directed graphs, it might only consider reachable nodes)
closeness_centrality = nx.closeness_centrality(G)
closeness_df = pd.DataFrame(closeness_centrality.items(), columns=['station', 'closeness'])

station_centralities = pd.merge(station_centralities, closeness_df, on='station')

print("Top 5 stations by Closeness Centrality:")
display(station_centralities.sort_values(by='closeness', ascending=False).head())

Top 5 stations by Closeness Centrality:


,station,in_deg,out_deg,betweenness,closeness
136,3000,0.979651,0.093023,0.003219,0.977045
41,3010,0.938953,0.915698,0.002697,0.936671
74,3185,0.924419,0.930233,0.003028,0.923945
104,3057,0.921512,0.930233,0.002763,0.921441
166,3255,0.909884,0.906977,0.002514,0.914010


#### d. PageRank Centrality

**Definition**: PageRank is an algorithm used to measure the relative importance of nodes within a network. It assigns a score to each node based on the number and quality of incoming links. A high PageRank score means a node is linked to by many other high-ranking nodes, indicating its overall importance or influence.

In [11]:
# Calculate PageRank Centrality
pagerank_centrality = nx.pagerank(G)
pagerank_df = pd.DataFrame(pagerank_centrality.items(), columns=['station', 'pagerank'])

station_centralities = pd.merge(station_centralities, pagerank_df, on='station')

print("Top 5 stations by PageRank Centrality:")
display(station_centralities.sort_values(by='pagerank', ascending=False).head())

Top 5 stations by PageRank Centrality:


,station,in_deg,out_deg,betweenness,closeness,pagerank
136,3000,0.979651,0.093023,0.003219,0.977045,0.027338
41,3010,0.938953,0.915698,0.002697,0.936671,0.013645
54,3032,0.892442,0.889535,0.002154,0.897128,0.011223
39,3054,0.904070,0.924419,0.002329,0.906698,0.009569
9,3028,0.889535,0.898256,0.001933,0.894767,0.009562


### 3. Merging Centrality Measures into the Trip Dataset

Finally, I'll merge these calculated centrality measures back into your original trip data (`fdemand`) to create a new dataset where each trip includes the centrality values of its starting and ending stations.

In [12]:
# Merge centrality measures for the start station
fdemand_with_centrality = pd.merge(
    fdemand,
    station_centralities.rename(columns={'station': 'start_station'}),
    on='start_station',
    how='left',
    suffixes=('', '_start')
)

# Merge centrality measures for the end station
fdemand_with_centrality = pd.merge(
    fdemand_with_centrality,
    station_centralities.rename(columns={'station': 'end_station'}),
    on='end_station',
    how='left',
    suffixes=('_start', '_end') # Ensure suffixes are applied correctly
)

# Rename columns for clarity (e.g., in_degree_centrality for start station becomes in_degree_centrality_start)
fdemand_with_centrality = fdemand_with_centrality.rename(columns={
    'in_deg': 'in_deg_start',
    'out_deg': 'out_deg_start',
    'betweenness': 'betweenness_start',
    'closeness': 'closeness_start',
    'pagerank': 'pagerank_start'
})

# Display the first few rows of the new dataset with centrality measures
print("New dataset with station centrality measures:")
display(fdemand_with_centrality.head())

New dataset with station centrality measures:


,duration,start_station,end_station,plan_duration,st_year,st_month,st_day,st_hour,st_minute,end_year,...,in_deg_start,out_deg_start,betweenness_start,closeness_start,pagerank_start,in_deg_end,out_deg_end,betweenness_end,closeness_end,pagerank_end
0,11,3049,3072,30,2020,1,1,0,13,2020,...,0.837209,0.863372,0.001581,0.856452,0.005218,0.793605,0.808140,0.001661,0.825271,0.003880
1,28,3124,3053,30,2020,1,1,0,14,2020,...,0.840116,0.848837,0.002198,0.856452,0.004823,0.837209,0.866279,0.001741,0.854301,0.005305
2,28,3124,3053,30,2020,1,1,0,14,2020,...,0.840116,0.848837,0.002198,0.856452,0.004823,0.837209,0.866279,0.001741,0.854301,0.005305
3,21,3005,3018,30,2020,1,1,0,17,2020,...,0.834302,0.837209,0.001171,0.852159,0.003923,0.898256,0.912791,0.001938,0.901888,0.006393
4,21,3005,3018,30,2020,1,1,0,17,2020,...,0.834302,0.837209,0.001171,0.852159,0.003923,0.898256,0.912791,0.001938,0.901888,0.006393


In [13]:
cols_to_merge = station_clean[['id', 'name', 'latitude', 'longitude']]

# Merge for start station's coordinates and name
fdemand_with_centrality = pd.merge(
    fdemand_with_centrality,
    cols_to_merge.rename(columns={'id': 'start_station', 'name': 'st_name', 'latitude': 'st_lat', 'longitude': 'st_lon'}),
    on='start_station',
    how='left'
)

# Merge for end station's coordinates and name
fdemand_with_centrality = pd.merge(
    fdemand_with_centrality,
    cols_to_merge.rename(columns={'id': 'end_station', 'name': 'end_name', 'latitude': 'end_lat', 'longitude': 'end_lon'}),
    on='end_station',
    how='left'
)

print("Dataset with start and end station geographical information:")
display(fdemand_with_centrality.head())

Dataset with start and end station geographical information:


,duration,start_station,end_station,plan_duration,st_year,st_month,st_day,st_hour,st_minute,end_year,...,out_deg_end,betweenness_end,closeness_end,pagerank_end,st_name,st_lat,st_lon,end_name,end_lat,end_lon
0,11,3049,3072,30,2020,1,1,0,13,2020,...,0.808140,0.001661,0.825271,0.003880,Foglietta Plaza,39.94509,-75.14250,Front & Carpenter,39.93445,-75.14541
1,28,3124,3053,30,2020,1,1,0,14,2020,...,0.866279,0.001741,0.854301,0.005305,Race Street Pier,39.95367,-75.13956,Point Breeze & Tasker,39.93231,-75.18154
2,28,3124,3053,30,2020,1,1,0,14,2020,...,0.866279,0.001741,0.854301,0.005305,Race Street Pier,39.95367,-75.13956,Point Breeze & Tasker,39.93231,-75.18154
3,21,3005,3018,30,2020,1,1,0,17,2020,...,0.912791,0.001938,0.901888,0.006393,"Welcome Park, NPS",39.94733,-75.14403,12th & Filbert,39.95273,-75.15979
4,21,3005,3018,30,2020,1,1,0,17,2020,...,0.912791,0.001938,0.901888,0.006393,"Welcome Park, NPS",39.94733,-75.14403,12th & Filbert,39.95273,-75.15979


### Adding Distance Between Stations

Calculating the geographical distance between the `start_station` and `end_station` is a strong predictor for trip duration and often for trip decisions. I will use the Haversine formula for this.

In [14]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in kilometers

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

# Apply the function to calculate distance for each trip
fdemand_with_centrality['dist_km'] = fdemand_with_centrality.apply(
    lambda row: haversine_distance(
        row['st_lat'], row['st_lon'], row['end_lat'], row['end_lon']
    ),
    axis=1
)

print("Dataset with calculated distance between stations:")
display(fdemand_with_centrality[['start_station', 'end_station', 'st_lat', 'st_lon', 'end_lat', 'end_lon', 'dist_km']].head())

Dataset with calculated distance between stations:


,start_station,end_station,st_lat,st_lon,end_lat,end_lon,dist_km
0,3049,3072,39.94509,-75.14250,39.93445,-75.14541,1.208846
1,3124,3053,39.95367,-75.13956,39.93231,-75.18154,4.295275
2,3124,3053,39.95367,-75.13956,39.93231,-75.18154,4.295275
3,3005,3018,39.94733,-75.14403,39.95273,-75.15979,1.471505
4,3005,3018,39.94733,-75.14403,39.95273,-75.15979,1.471505


### Merging Weather Data

To enrich the trip data with weather conditions, I will merge the `weather` dataset using the trip's start date. First, I need to ensure both `fdemand_with_centrality` and `weather` have a common date column in datetime format.

In [15]:
# Convert 'date' column in weather DataFrame to datetime objects
weather['date'] = pd.to_datetime(weather['date'])

# Create a 'start_date' column in fdemand_with_centrality from existing date components
fdemand_with_centrality['start_date'] = pd.to_datetime(
    fdemand_with_centrality[['st_year', 'st_month', 'st_day']].rename(
        columns={'st_year': 'year', 'st_month': 'month', 'st_day': 'day'})
)

# Perform a left merge to add weather data to fdemand_with_centrality
fdemand_with_centrality = pd.merge(
    fdemand_with_centrality,
    weather,
    left_on='start_date',
    right_on='date',
    how='left',
    #suffixes=('', '_weather') # Add suffix to weather columns to avoid conflicts
)

# Drop the redundant 'date_weather' column from the merge
fdemand_with_centrality = fdemand_with_centrality.drop(columns=['date'])

print("Dataset with merged weather information:")
display(fdemand_with_centrality.head())

Dataset with merged weather information:


,duration,start_station,end_station,plan_duration,st_year,st_month,st_day,st_hour,st_minute,end_year,...,end_lat,end_lon,dist_km,start_date,avg_wind,precip,snow,snow_ground,max_temp,min_temp
0,11,3049,3072,30,2020,1,1,0,13,2020,...,39.93445,-75.14541,1.208846,2020-01-01,9.17,0.0,0.0,0.0,42,29
1,28,3124,3053,30,2020,1,1,0,14,2020,...,39.93231,-75.18154,4.295275,2020-01-01,9.17,0.0,0.0,0.0,42,29
2,28,3124,3053,30,2020,1,1,0,14,2020,...,39.93231,-75.18154,4.295275,2020-01-01,9.17,0.0,0.0,0.0,42,29
3,21,3005,3018,30,2020,1,1,0,17,2020,...,39.95273,-75.15979,1.471505,2020-01-01,9.17,0.0,0.0,0.0,42,29
4,21,3005,3018,30,2020,1,1,0,17,2020,...,39.95273,-75.15979,1.471505,2020-01-01,9.17,0.0,0.0,0.0,42,29


In [16]:
data = fdemand_with_centrality.drop(['start_date'], axis = 1)

In [17]:
data.head()

,duration,start_station,end_station,plan_duration,st_year,st_month,st_day,st_hour,st_minute,end_year,...,end_name,end_lat,end_lon,dist_km,avg_wind,precip,snow,snow_ground,max_temp,min_temp
0,11,3049,3072,30,2020,1,1,0,13,2020,...,Front & Carpenter,39.93445,-75.14541,1.208846,9.17,0.0,0.0,0.0,42,29
1,28,3124,3053,30,2020,1,1,0,14,2020,...,Point Breeze & Tasker,39.93231,-75.18154,4.295275,9.17,0.0,0.0,0.0,42,29
2,28,3124,3053,30,2020,1,1,0,14,2020,...,Point Breeze & Tasker,39.93231,-75.18154,4.295275,9.17,0.0,0.0,0.0,42,29
3,21,3005,3018,30,2020,1,1,0,17,2020,...,12th & Filbert,39.95273,-75.15979,1.471505,9.17,0.0,0.0,0.0,42,29
4,21,3005,3018,30,2020,1,1,0,17,2020,...,12th & Filbert,39.95273,-75.15979,1.471505,9.17,0.0,0.0,0.0,42,29


In [18]:
data.to_csv('fdemand.csv', index = False)

KeyboardInterrupt: 